In [2]:
!pip install -U peft datasets sacrebleu sentencepiece accelerate evaluate bitsandbytes transformers==4.44.2 huggingface_hub==0.23.5

  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached datasets-4.8.3-py3-none-any.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of peft to determine which version is compatible with other requirements. This could take a while.
  Using cached peft-0.18.0-py3-none-any.whl.metadata (14 kB)
  Using cached peft-0.17.1-py3-none-any.whl.metadata (14 kB)
  Using cached peft-0.17.0-py3-none-any.whl.metadata (14 kB)
  Using cached peft-0.16.0-py3-none-any.whl.metadata (14 kB)
  Using cached peft-0.15.2-py3-none-any.whl.metadata (13 kB)
  Using cached peft-0.15.1-py3-none-any.whl.metadata (13 kB)
  Using cached peft-0.15.0-py3-none-any.whl.metadata (13 kB)
INFO: pip is still looking at multiple versions of peft to determine which version is compatible with other requirements. This could take a while.
  Using cached peft-0.14.0-py3-none-any.whl.metadata (13 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
INFO: pip is 

In [3]:
import transformers
print(transformers.__version__)

4.44.2


In [4]:
from datasets import load_dataset
import random

ds = load_dataset("nhuvo/MedEV")
dataset = ds["train"]

offset = 340897
num_pairs = 340897

pairs = []

for i in range(num_pairs):
    en = dataset[i]["text"]
    vi = dataset[i + offset]["text"]

    pairs.append({
        "en": en,
        "vi": vi
    })


random.seed(42)
random.shuffle(pairs)
pairs_30k = pairs[:30000]

train_size = int(0.8 * len(pairs_30k))
val_size = int(0.1 * len(pairs_30k))

train_pairs = pairs_30k[:train_size]
val_pairs = pairs_30k[train_size:train_size+val_size]
test_pairs = pairs_30k[train_size+val_size:]

README.md: 0.00B [00:00, ?B/s]

train.en.txt:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

train.vi.txt:   0%|          | 0.00/61.9M [00:00<?, ?B/s]

val.en.new.txt: 0.00B [00:00, ?B/s]

val.vi.new.txt: 0.00B [00:00, ?B/s]

test.en.new.txt: 0.00B [00:00, ?B/s]

test.vi.new.txt: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/681794 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17878 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17920 [00:00<?, ? examples/s]

In [5]:
import torch
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# ===== LOAD MODEL =====
model_name = "VietAI/envit5-translation"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    low_cpu_mem_usage=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

# ===== METRICS =====
bleu = evaluate.load("bleu")
ter = evaluate.load("ter")
meteor = evaluate.load("meteor")

# ===== PREPARE DATA =====
batch_size = 32   # thử 16 / 32 / 64 tùy GPU

predictions = []
references = []

total = len(test_pairs)

print(f"Start batch inference on {total} samples...\n")

# ===== BATCH LOOP =====
for i in tqdm(range(0, total, batch_size)):

    batch = test_pairs[i:i+batch_size]

    src_batch = [
        "translate English to Vietnamese: " + x["en"]
        for x in batch
    ]

    tgt_batch = [x["vi"] for x in batch]

    # tokenize batch
    inputs = tokenizer(
        src_batch,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda:0")

    # generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    predictions.extend(preds)
    references.extend([[t] for t in tgt_batch])

# ===== METRICS =====
print("\nComputing metrics...\n")

bleu_result = bleu.compute(predictions=predictions, references=references)

ter_result = ter.compute(
    predictions=predictions,
    references=[r[0] for r in references]
)

meteor_result = meteor.compute(
    predictions=predictions,
    references=[r[0] for r in references]
)

# ===== RESULT =====
print("===== FINAL RESULTS =====")
print(f"BLEU   : {bleu_result['bleu']:.4f}")
print(f"TER    : {ter_result['score']:.4f}")
print(f"METEOR : {meteor_result['meteor']:.4f}")

2026-03-20 02:24:10.356852: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773973450.784191      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773973450.897623      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773973451.770582      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773973451.770627      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773973451.770631      55 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

...

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Start batch inference on 3000 samples...



100%|██████████| 94/94 [03:48<00:00,  2.43s/it]



Computing metrics...

===== FINAL RESULTS =====
BLEU   : 0.4084
TER    : 54.4297
METEOR : 0.6491
